# Lab 2 Exercise: three tools in a LangGraph loop

Three tools find Stockholm's temperature, convert it to Fahrenheit, and send both values by push notification. Each step needs the last result, so the graph runs the tool loop three times.

## Imports and environment

Add `OPENAI_API_KEY`, `SERPER_API_KEY`, `PUSHOVER_TOKEN`, and `PUSHOVER_USER` to `.env` before running the notebook.

In [ ]:
import os
import requests
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(override=True)

## Three tools

Search and push notification come from the lab. The new tool converts Celsius to Fahrenheit.

In [ ]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from degrees Celsius to degrees Fahrenheit."""
    return round((celsius * 9 / 5) + 32, 1)

@tool
def send_push_notification(text: str) -> str:
    """Send a short push notification to the user's phone."""
    requests.post(
        "https://api.pushover.net/1/messages.json",
        data={
            "token": os.getenv("PUSHOVER_TOKEN"),
            "user": os.getenv("PUSHOVER_USER"),
            "message": text,
        },
    )
    return "Notification sent"

tools = [search, celsius_to_fahrenheit, send_push_notification]

for available_tool in tools:
    print(available_tool.name)

## Bind the tools to the model

Bind all three tools to the model. The chatbot adds the model's reply to the graph state.

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini")
llm_with_tools = llm.bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot_node(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Build the graph

`tools_condition` routes tool requests to `tools`. The graph then returns to `chatbot`. It stops when the model requests no tool.

In [ ]:
builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

## Run and trace the graph

Stream each graph update to show the node order, tool calls, and tool results. If LangSmith tracing is on, open this run in LangSmith and confirm the same node order.

In [ ]:
question = """
Use your tools in this exact order, waiting for the result of each tool before requesting the next one:
1. Search for the current temperature in Stockholm in degrees Celsius.
2. Pass that Celsius value to the celsius_to_fahrenheit tool.
3. Send a push notification containing both the Celsius and Fahrenheit temperatures.
After the notification is sent, give me a brief confirmation.
"""

tool_request_rounds = 0
node_order = []
final_answer = ""

inputs = {"messages": [{"role": "user", "content": question}]}
for update in graph.stream(inputs, stream_mode="updates"):
    for node_name, node_result in update.items():
        node_order.append(node_name)

        for message in node_result.get("messages", []):
            if getattr(message, "tool_calls", None):
                tool_request_rounds += 1
                for call in message.tool_calls:
                    print(f"AI requested {call['name']} with {call['args']}")
            elif message.type == "tool":
                print(f"Tool {message.name} returned: {message.content}")
            elif node_name == "chatbot":
                final_answer = message.content

print("\nNode order:", " -> ".join(node_order))
print(f"Tools node visits: {node_order.count('tools')}")
print(f"\nTool-request rounds: {tool_request_rounds}")
print("Final answer:")
print(final_answer)